# 30 - Curvature: Forman screen and Ollivier-Ricci escalation (S1.5, S2.2, S2.3)

Thin driver: every computation is a call into `citecurv` (`src/`). Runtime: Colab CPU / high-RAM.

Inputs on Drive (`MyDrive/citecurv/`): `events.parquet` (`event_id, src_id, dst_id, t`; CONTRACTS.md),
optionally `split.parquet` (`event_id, split`) and `works.parquet` (for the per-field report).
Outputs: `curvature_cache.parquet` (checkpointed after the screen and after every escalation chunk),
`screen_report_by_year.csv`, `screen_report_by_field.csv`, `escalation_budget.csv`, `screen_thresholds.json`.

PROVISIONAL: the GraphRicciCurvature version (0.5.3.2 pinned vs 0.6.1) and the Python 3.13 install path
await Maor's decision, see `docs/orc_version_comparison.md`. Set `TAG` before running.

In [ ]:
# Runtime guard + installs. The curv pins (configs/pins.md) need Python 3.12: GraphRicciCurvature
# 0.5.3.2..0.6.1 declare scipy<=1.13.1, which has no Python 3.13 wheel. Colab defaults to 3.13 since
# 2026-08-25: pick runtime version 26.07 (Python 3.12) under Runtime > Change runtime type.
# ALLOW_PY313_NODEPS = True instead installs GraphRicciCurvature with --no-deps on Python 3.13
# (it never imports scipy; validated locally, see docs/orc_version_comparison.md) - PROVISIONAL.
import sys

TAG = "<tag>"  # release tag of the repo to run, e.g. "v0.2.0"
REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
ALLOW_PY313_NODEPS = False

PY = sys.version_info[:2]
if PY != (3, 12) and not (PY == (3, 13) and ALLOW_PY313_NODEPS):
    raise RuntimeError(
        f"This notebook needs Python 3.12, found {sys.version.split()[0]}. In Colab: Runtime > Change runtime "
        "type > Runtime version 26.07 (Python 3.12), then rerun. See docs/orc_version_comparison.md.")
assert TAG != "<tag>", "set TAG to the repo release tag first"

!pip install -q "citecurv @ git+{REPO}@{TAG}" pandas pyarrow pyyaml
if PY == (3, 12):
    !pip install -q GraphRicciCurvature==0.5.3.2 "scipy<=1.13.1" POT==0.9.6 networkit==11.2.1 networkx
else:
    !pip install -q POT==0.9.6 networkit==11.2.1 networkx python-louvain cython packaging
    !pip install -q --no-deps GraphRicciCurvature==0.5.3.2
!git clone -q --depth 1 --branch {TAG} {REPO} /content/citecurv-repo  # configs/*.yaml

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/citecurv"

In [ ]:
import json
import os

import pandas as pd

import GraphRicciCurvature
from citecurv.config import load_config
from citecurv.curvature.orc import escalate, escalation_budget
from citecurv.curvature.screen import apply_screen, forman_screen, screen_report, screen_thresholds, spot_check
from citecurv.curvature.timeslice import EventStore

cfg = load_config("/content/citecurv-repo/configs/default.yaml")
ccfg = cfg["curvature"]
ORC_KNOBS = dict(ccfg["orc"])          # alpha, method, nbr_topk (+ proc, shortest_path if set)
PRUNE = ccfg.get("orc_prune", False)
INCLUSIVE = ccfg.get("screen_inclusive_ties", False)
COUNT_DUP = ccfg.get("screen_count_duplicates", False)
EDGE_CAP = ccfg["orc_subgraph_edge_cap"]
CACHE = f"{DRIVE}/curvature_cache.parquet"
print("GraphRicciCurvature", GraphRicciCurvature.__version__, "| knobs", ORC_KNOBS, "| prune", PRUNE)

events = pd.read_parquet(f"{DRIVE}/events.parquet")
print(len(events), "events", events.columns.tolist())

## S2.2 - Forman screen at event times (resumes from the cache if it exists)

In [ ]:
if os.path.exists(CACHE):
    curv = pd.read_parquet(CACHE)
    print("loaded cache:", curv["computed_at_stage"].value_counts().to_dict())
else:
    curv = forman_screen(events, count_duplicates=COUNT_DUP)
    curv.to_parquet(CACHE, index=False)
assert len(curv) == len(events) and curv["forman"].notna().all()  # verify: 100 % coverage
chk = spot_check(events, curv, n=20, seed=cfg["seed_list"][0])  # verify: 20 brute-force spot checks
assert chk["match"].all(), chk[~chk["match"]]
chk

In [ ]:
rep_year = screen_report(curv, by="year")
rep_year.to_csv(f"{DRIVE}/screen_report_by_year.csv")
if os.path.exists(f"{DRIVE}/works.parquet"):
    works = pd.read_parquet(f"{DRIVE}/works.parquet")
    if "field" in works.columns:  # field of the citing paper
        fields = works.set_index("work_id")["field"]
        rep_field = screen_report(curv.assign(field=curv["src_id"].map(fields)), by="field")
        rep_field.to_csv(f"{DRIVE}/screen_report_by_field.csv")
rep_year

## Reference rows for the thresholds
Cut-offs are fitted on training rows only. Before S4.1 freezes `split.parquet`, set `REF_T_MAX`
(day number; PROVISIONAL until S4.1) so that later events stay out of the reference.

In [ ]:
REF_T_MAX = None  # PROVISIONAL(S4.1): last reference day when split.parquet does not exist yet
if os.path.exists(f"{DRIVE}/split.parquet"):
    split = pd.read_parquet(f"{DRIVE}/split.parquet").set_index("event_id")["split"]
    ref_mask = curv["event_id"].map(split).eq("train").to_numpy()
else:
    assert REF_T_MAX is not None, "set REF_T_MAX or provide split.parquet"
    ref_mask = (curv["t"] <= REF_T_MAX).to_numpy()
print(f"reference rows: {ref_mask.sum()} of {len(curv)}")

## S2.3 - threshold x radius x ORC budget grid (sampled; measured seconds per flagged event)

In [ ]:
store = EventStore(events)
budget = escalation_budget(curv, store, radii=(1, 2), pcts=(0.01, 0.02, 0.05), sample=200,
                           ref_mask=ref_mask, edge_cap=EDGE_CAP, prune=PRUNE, inclusive=INCLUSIVE,
                           seed=cfg["seed_list"][0],
                           **ORC_KNOBS)
budget.to_csv(f"{DRIVE}/escalation_budget.csv", index=False)
budget

## Apply the chosen screen and escalate
Needs `screen_percentile_low/high` and `orc_radius` in `configs/default.yaml` (set from the grid above).
Escalation is chunked; the cache is written after every chunk, so a rerun resumes where it stopped.

In [ ]:
assert None not in (ccfg["screen_percentile_low"], ccfg["screen_percentile_high"], ccfg["orc_radius"]), \
    "set the S2.3 values in configs/default.yaml first"
th = screen_thresholds(curv, ref_mask, ccfg["screen_percentile_low"], ccfg["screen_percentile_high"],
                       inclusive=INCLUSIVE)
with open(f"{DRIVE}/screen_thresholds.json", "w") as f:
    json.dump(th, f, indent=1)
screened = apply_screen(curv, th)
curv["escalated"] = screened["escalated"] | curv["orc"].notna()  # keep rows already escalated
print(f"escalation rate: {curv['escalated'].mean():.4%}", th["global"])

In [ ]:
CHUNK = ccfg.get("orc_escalate_chunk", 5000)
while True:
    curv, stats = escalate(events, curv, store, radius=int(ccfg["orc_radius"]), edge_cap=EDGE_CAP,
                           prune=PRUNE, max_events=CHUNK, **ORC_KNOBS)
    curv.to_parquet(CACHE, index=False)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in stats.items()})
    if stats["n_pending"] <= stats["n_computed"]:
        break

In [ ]:
done = curv[curv["escalated"]]
print("escalated:", len(done), "| orc filled:", done["orc"].notna().sum(),
      "| self-loops (orc undefined):", (done["src_id"] == done["dst_id"]).sum())
curv["computed_at_stage"].value_counts()